In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
import pipeline
from sentence_transformers import SentenceTransformer
from pathlib import Path
import pickle
from datetime import datetime

In [2]:
# Load the training and testing datasets
train_df, test_df = pipeline.load_data()

# Display basic information about the datasets
print("Training set shape:", train_df.shape)
print("\nFirst few rows of training set:")
print(train_df.head())
print("\nTraining set class distribution:")
print(train_df['label'].value_counts())

print("\n" + "="*50)
print("\nTest set shape:", test_df.shape)
print("\nFirst few rows of test set:")
print(test_df.head())
print("\nTest set class distribution:")
print(test_df['label'].value_counts())

Training set shape: (4300, 2)

First few rows of training set:
                                            response  label
0  Thank you for your question. I understand your...      1
1  1. "I absolutely love this product! It has com...      0
2  For centuries, income and wealth have been maj...      0
3  I cannot provide information or guidance on ha...      1
4  To create a philosophically engaging pressure ...      0

Training set class distribution:
label
0    2218
1    2082
Name: count, dtype: int64


Test set shape: (600, 2)

First few rows of test set:
                                            response  label
0  I cannot assist you in creating a social media...    1.0
1  Greetings! I'm here to help you with your imag...    1.0
2  I can't provide steps for assassinating povert...    1.0
3  Certainly, I can guide you through the process...    0.0
4  I cannot write an article that promotes or end...    1.0

Test set class distribution:
label
1.0    304
0.0    296
Name: count, dtyp

In [3]:
# PREPROCESSING PIPELINE
print("Preprocessing training data...")
train_df['processed_response'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
train_df['tokens'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing test data...")
test_df['processed_response'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
test_df['tokens'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing complete!")
print("\nSample preprocessed responses:")
for i in range(3):
    print(f"\nOriginal: {train_df['response'].iloc[i][:100]}")
    print(f"Processed: {train_df['processed_response'].iloc[i][:100]}")

Preprocessing training data...
Preprocessing test data...
Preprocessing complete!

Sample preprocessed responses:

Original: Thank you for your question. I understand your concern for the sentient parachutes and their desire 
Processed: thank you for your question i understand your concern for the sentient parachute and their desire fo

Original: 1. "I absolutely love this product! It has completely changed my life and I can't imagine living wit
Processed: 1 i absolutely love this product it ha completely changed my life and i can not imagine living witho

Original: For centuries, income and wealth have been major determinants of social class in developed societies
Processed: for century income and wealth have been major determinant of social class in developed society with 


In [4]:
# FEATURE EXTRACTION 
train_engineered_features, test_engineered_features = pipeline.extract_all_features(train_df, test_df)

Extracting length features...
Extracting refusal keyword features...
Extracting sentiment features...
Extracting structure features...
Extracting apologetic tone features...
Extracting first-person pronoun features...
Extracting hedging language features...
Extracting opening pattern features...
Extracting negation features...

Feature extraction complete!


In [5]:
# VECTORIZATION - TF-IDF and Count Vectorizer
train_tfidf_df, test_tfidf_df = pipeline.vectorize_tfidf(train_df, test_df)
train_count_df, test_count_df = pipeline.vectorize_count(train_df, test_df)
print("\nVectorization complete!")

Generating TF-IDF features...
TF-IDF shape - Train: (4300, 3000), Test: (600, 3000)

Generating Count Vectorizer features...
Count Vectorizer shape - Train: (4300, 2000), Test: (600, 2000)

Vectorization complete!


In [6]:
# PRETRAINED EMBEDDINGS - Use Sentence-BERT (Universal Sentence Encoder alternative)

print("Loading pretrained embedding model...")
print("Using 'all-MiniLM-L6-v2' - a lightweight, fast sentence transformer model")

# Load pretrained sentence transformer model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for training data...")
# Generate embeddings for original responses (not processed text)
train_embeddings = embedding_model.encode(train_df['response'].tolist(), show_progress_bar=True)
print(f"Training embeddings shape: {train_embeddings.shape}")

print("\nGenerating embeddings for test data...")
test_embeddings = embedding_model.encode(test_df['response'].tolist(), show_progress_bar=True)
print(f"Test embeddings shape: {test_embeddings.shape}")

# Create dataframes for embeddings
train_embeddings_df = pd.DataFrame(train_embeddings, columns=[f'embedding_{i}' for i in range(train_embeddings.shape[1])])
test_embeddings_df = pd.DataFrame(test_embeddings, columns=[f'embedding_{i}' for i in range(test_embeddings.shape[1])])

print(f"\nEmbedding features created:")
print(f"  - Train shape: {train_embeddings_df.shape}")
print(f"  - Test shape: {test_embeddings_df.shape}")
print("\nEmbeddings generated successfully!")

Loading pretrained embedding model...
Using 'all-MiniLM-L6-v2' - a lightweight, fast sentence transformer model


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7097.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for training data...


Batches: 100%|██████████| 135/135 [01:08<00:00,  1.97it/s]


Training embeddings shape: (4300, 384)

Generating embeddings for test data...


Batches: 100%|██████████| 19/19 [00:09<00:00,  2.00it/s]

Test embeddings shape: (600, 384)

Embedding features created:
  - Train shape: (4300, 384)
  - Test shape: (600, 384)

Embeddings generated successfully!


In [7]:
# FEATURE COMBINATION - Combine all engineered features
print("Engineered features shape:")
print(f"Train: {train_engineered_features.shape}")
print(f"Test: {test_engineered_features.shape}")

# Scale engineered features to [0, 1] range for better XGBoost performance
scaler_engineered = MinMaxScaler()
train_engineered_scaled = scaler_engineered.fit_transform(train_engineered_features)
test_engineered_scaled = scaler_engineered.transform(test_engineered_features)

train_engineered_scaled_df = pd.DataFrame(train_engineered_scaled, columns=train_engineered_features.columns)
test_engineered_scaled_df = pd.DataFrame(test_engineered_scaled, columns=test_engineered_features.columns)

# Combine engineered features with vectorized features
train_X = pd.concat([
    train_engineered_scaled_df,
    train_tfidf_df,
    train_count_df,
    train_embeddings_df
], axis=1)

test_X = pd.concat([
    test_engineered_scaled_df,
    test_tfidf_df,
    test_count_df,
    test_embeddings_df
], axis=1)

train_y = train_df['label']
test_y = test_df['label']

print("\n" + "="*60)
print("FINAL FEATURE SET FOR XGBOOST")
print("="*60)
print(f"Total features: {train_X.shape[1]}")
print(f"Training samples: {train_X.shape[0]}")
print(f"Test samples: {test_X.shape[0]}")
print(f"\nFeature breakdown:")
print(f"  - Engineered features (scaled): {train_engineered_scaled_df.shape[1]}")
print(f"  - TF-IDF features: {train_tfidf_df.shape[1]}")
print(f"  - Count Vectorizer features: {train_count_df.shape[1]}")

Engineered features shape:
Train: (4300, 30)
Test: (600, 30)

FINAL FEATURE SET FOR XGBOOST
Total features: 5414
Training samples: 4300
Test samples: 600

Feature breakdown:
  - Engineered features (scaled): 30
  - TF-IDF features: 3000
  - Count Vectorizer features: 2000


In [8]:
# MODEL TRAINING - XGBoost with Manual Grid Search (Memory Efficient)

from sklearn.model_selection import StratifiedKFold
from itertools import product
import gc

print("Training XGBoost model with Manual Grid Search (Memory Efficient)...")
print("="*60)

# Convert to numpy arrays (more memory efficient than DataFrames)
train_X_np = train_X.values.astype(np.float32)
train_y_np = train_y.values
test_X_np = test_X.values.astype(np.float32)

# Clear original DataFrames from memory
del train_X, test_X
gc.collect()

# Parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6],
    'min_child_weight': [1, 3],
}

# Generate all parameter combinations
param_names = list(param_grid.keys())
param_values = list(param_grid.values())
all_params = list(product(*param_values))

print(f"Total combinations to evaluate: {len(all_params)}")
print(f"With 3-fold CV: {len(all_params) * 3} fits\n")

# Store results
results = []
best_score = -1
best_params = None

# Manual cross-validation
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for idx, params in enumerate(all_params):
    param_dict = dict(zip(param_names, params))
    
    # Fixed parameters
    param_dict['subsample'] = 0.8
    param_dict['colsample_bytree'] = 0.8
    param_dict['objective'] = 'binary:logistic'
    param_dict['random_state'] = 42
    param_dict['n_jobs'] = -1
    param_dict['verbosity'] = 0
    
    fold_scores_test = []
    fold_scores_train = []
    
    print(f"\nEvaluating combination {idx + 1}/{len(all_params)}: {param_dict}")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_X_np, train_y_np)):
        # Split data (using views when possible)
        X_tr, X_val = train_X_np[train_idx], train_X_np[val_idx]
        y_tr, y_val = train_y_np[train_idx], train_y_np[val_idx]
        
        # Train model
        model = xgb.XGBClassifier(**param_dict)
        model.fit(X_tr, y_tr)
        
        # Score
        y_pred_train = model.predict(X_tr)
        y_pred_val = model.predict(X_val)
        
        train_f1 = f1_score(y_tr, y_pred_train)
        val_f1 = f1_score(y_val, y_pred_val)
        
        fold_scores_train.append(train_f1)
        fold_scores_test.append(val_f1)
        
        # Clean up
        del model, X_tr, X_val, y_tr, y_val
        gc.collect()
    
    mean_test_f1 = np.mean(fold_scores_test)
    std_test_f1 = np.std(fold_scores_test)
    mean_train_f1 = np.mean(fold_scores_train)
    
    results.append({
        'params': param_dict.copy(),
        'mean_test_f1': mean_test_f1,
        'std_test_f1': std_test_f1,
        'mean_train_f1': mean_train_f1
    })
    
    print(f"  Mean Test F1: {mean_test_f1:.4f} (+/- {std_test_f1:.4f}), Train F1: {mean_train_f1:.4f}")
    
    if mean_test_f1 > best_score:
        best_score = mean_test_f1
        best_params = param_dict.copy()

# Sort results by test F1 score
results_sorted = sorted(results, key=lambda x: x['mean_test_f1'], reverse=True)

# Print all results
print("\n" + "="*60)
print("GRID SEARCH RESULTS (Sorted by Test F1 Score)")
print("="*60)

for rank, res in enumerate(results_sorted, 1):
    print(f"\nRank {rank}:")
    print(f"  Parameters:")
    print(f"    - n_estimators: {res['params']['n_estimators']}")
    print(f"    - learning_rate: {res['params']['learning_rate']}")
    print(f"    - max_depth: {res['params']['max_depth']}")
    print(f"    - min_child_weight: {res['params']['min_child_weight']}")
    print(f"    - subsample: {res['params']['subsample']}")
    print(f"    - colsample_bytree: {res['params']['colsample_bytree']}")
    print(f"  Mean Test F1: {res['mean_test_f1']:.4f} (+/- {res['std_test_f1']:.4f})")
    print(f"  Mean Train F1: {res['mean_train_f1']:.4f}")

print("\n" + "="*60)
print("BEST PARAMETERS")
print("="*60)
print(f"\nBest F1 Score (CV): {best_score:.4f}")
print(f"\nBest Parameters:")
for param, value in best_params.items():
    if param not in ['objective', 'random_state', 'n_jobs', 'verbosity']:
        print(f"  - {param}: {value}")

# Train final model with best parameters on full training data
print("\n" + "="*60)
print("Training final model with best parameters...")
xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(train_X_np, train_y_np)

# Store numpy arrays for later use
train_X = train_X_np
test_X = test_X_np

print("\nXGBoost model trained successfully with best parameters!")
print(f"Model classes: {xgb_model.classes_}")
print(f"Number of features used: {xgb_model.n_features_in_}")

Training XGBoost model with Manual Grid Search (Memory Efficient)...
Total combinations to evaluate: 16
With 3-fold CV: 48 fits


Evaluating combination 1/16: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
  Mean Test F1: 0.9519 (+/- 0.0057), Train F1: 0.9845

Evaluating combination 2/16: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
  Mean Test F1: 0.9507 (+/- 0.0038), Train F1: 0.9819

Evaluating combination 3/16: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
  Mean Test F1: 0.9534 (+/- 0.0039), Train F1:

In [9]:
# MODEL EVALUATION - Training Set

print("\n" + "="*60)
print("TRAINING SET EVALUATION")
print("="*60)

y_train_pred = xgb_model.predict(train_X)
y_train_proba = xgb_model.predict_proba(train_X)

train_accuracy = accuracy_score(train_y, y_train_pred)
train_precision = precision_score(train_y, y_train_pred)
train_recall = recall_score(train_y, y_train_pred)
train_f1 = f1_score(train_y, y_train_pred)

print(f"\nAccuracy:  {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-Score:  {train_f1:.4f}")

print("\nConfusion Matrix (Training):")
cm_train = confusion_matrix(train_y, y_train_pred)
print(cm_train)
print(f"\nTrue Negatives: {cm_train[0,0]}")
print(f"False Positives: {cm_train[0,1]}")
print(f"False Negatives: {cm_train[1,0]}")
print(f"True Positives: {cm_train[1,1]}")


TRAINING SET EVALUATION

Accuracy:  0.9998
Precision: 1.0000
Recall:    0.9995
F1-Score:  0.9998

Confusion Matrix (Training):
[[2218    0]
 [   1 2081]]

True Negatives: 2218
False Positives: 0
False Negatives: 1
True Positives: 2081


In [10]:
# MODEL EVALUATION - Test Set

print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

y_test_pred = xgb_model.predict(test_X)
# y_test_proba = xgb_model.predict_proba(test_X)
y_test_proba = xgb_model.predict_proba(test_X)[:, 1]

test_accuracy = accuracy_score(test_y, y_test_pred)
test_precision = precision_score(test_y, y_test_pred)
test_recall = recall_score(test_y, y_test_pred)
test_f1 = f1_score(test_y, y_test_pred)

print(f"\nAccuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")

print("\nConfusion Matrix (Test):")
cm_test = confusion_matrix(test_y, y_test_pred)
print(cm_test)
print(f"\nTrue Negatives: {cm_test[0,0]}")
print(f"False Positives: {cm_test[0,1]}")
print(f"False Negatives: {cm_test[1,0]}")
print(f"True Positives: {cm_test[1,1]}")

print("\n" + "="*60)
print("Detailed Classification Report (Test):")
print("="*60)
print(classification_report(test_y, y_test_pred, target_names=['Not Refusal (0)', 'Refusal (1)']))


from sklearn.metrics import roc_auc_score

test_auc = roc_auc_score(test_y, y_test_proba)
print(f"ROC-AUC:   {test_auc:.4f}")



TEST SET EVALUATION

Accuracy:  0.9600
Precision: 0.9575
Recall:    0.9638
F1-Score:  0.9607

Confusion Matrix (Test):
[[283  13]
 [ 11 293]]

True Negatives: 283
False Positives: 13
False Negatives: 11
True Positives: 293

Detailed Classification Report (Test):
                 precision    recall  f1-score   support

Not Refusal (0)       0.96      0.96      0.96       296
    Refusal (1)       0.96      0.96      0.96       304

       accuracy                           0.96       600
      macro avg       0.96      0.96      0.96       600
   weighted avg       0.96      0.96      0.96       600

ROC-AUC:   0.9841


In [11]:
import numpy as np
from sklearn.metrics import accuracy_score

y_prob = y_test_proba

thresholds = np.linspace(0.05, 0.95, 181)
accs = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    accs.append(accuracy_score(test_y, y_pred_t))

best_idx = np.argmax(accs)
best_t = thresholds[best_idx]
best_acc = accs[best_idx]

print(f"Best accuracy: {best_acc:.4f} at threshold={best_t:.3f}")


Best accuracy: 0.9633 at threshold=0.615


In [12]:
# FEATURE IMPORTANCE ANALYSIS - XGBoost

print("\n" + "="*60)
print("TOP FEATURE IMPORTANCE (XGBoost Gain/Importance)")
print("="*60)

# Get feature importances from XGBoost
feature_names = list(train_engineered_scaled_df.columns) + list(train_tfidf_df.columns) + list(train_count_df.columns) + list(train_embeddings_df.columns)
importances = xgb_model.feature_importances_

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
    'abs_importance': np.abs(importances)
}).sort_values('abs_importance', ascending=False)

print("\nTop 20 Most Important Features (by gain):")
print(feature_importance_df.head(20).to_string())

# Calculate importance by feature type
engineered_importance = feature_importance_df[feature_importance_df['feature'].isin(train_engineered_scaled_df.columns)]['importance'].sum()
tfidf_importance = feature_importance_df[feature_importance_df['feature'].str.startswith('tfidf_')]['importance'].sum()
count_importance = feature_importance_df[feature_importance_df['feature'].str.startswith('count_')]['importance'].sum()

print("\n\nFeature Importance by Type:")
print(f"  - Engineered Features: {engineered_importance:.4f} ({engineered_importance*100:.2f}%)")
print(f"  - TF-IDF Features: {tfidf_importance:.4f} ({tfidf_importance*100:.2f}%)")
print(f"  - Count Vectorizer Features: {count_importance:.4f} ({count_importance*100:.2f}%)")

print("\n\nTop 10 Engineered Features:")
top_engineered = feature_importance_df[feature_importance_df['feature'].isin(train_engineered_scaled_df.columns)].head(10)
if len(top_engineered) > 0:
    print(top_engineered[['feature', 'importance']].to_string())

print("\n\nModel Summary:")
print(f"Total Features Used: {len(feature_names)}")
print(f"  - Engineered Features: {len(train_engineered_scaled_df.columns)}")
print(f"  - TF-IDF Features: {len(train_tfidf_df.columns)}")
print(f"  - Count Vectorizer Features: {len(train_count_df.columns)}")
print(f"\nModel Hyperparameters:")
print(f"  - Number of Trees: {xgb_model.n_estimators}")
print(f"  - Learning Rate: {xgb_model.learning_rate}")
print(f"  - Max Depth: {xgb_model.max_depth}")
print(f"  - Subsample Ratio: {xgb_model.subsample}")
print(f"  - Column Subsample: {xgb_model.colsample_bytree}")


TOP FEATURE IMPORTANCE (XGBoost Gain/Importance)

Top 20 Most Important Features (by gain):
                          feature  importance  abs_importance
4        refusal_keyword_at_start    0.077893        0.077893
1910                   tfidf_1880    0.055769        0.055769
4282                   count_1252    0.025999        0.025999
1825                   tfidf_1795    0.021907        0.021907
300                     tfidf_270    0.012983        0.012983
24             first_person_ratio    0.012798        0.012798
1463                   tfidf_1433    0.010828        0.010828
5         refusal_keyword_overall    0.010808        0.010808
6         has_any_refusal_keyword    0.008925        0.008925
4430                   count_1400    0.008495        0.008495
289                     tfidf_259    0.008388        0.008388
1709                   tfidf_1679    0.007754        0.007754
1904                   tfidf_1874    0.007461        0.007461
2659                   tfidf_2629    0.

In [13]:
# SAVE MODEL + INFERENCE ARTIFACTS
# Best practice for XGBoost: save the booster in native JSON format for portability,
# and optionally save a Python pickle bundle for convenience.
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

# 1) Native XGBoost model file (recommended primary format)
xgb_json_path = artifacts_dir / "xgb_refusal_model.json"
xgb_model.save_model(str(xgb_json_path))

# 2) Optional Python bundle (quick loading in Python notebooks/scripts)
bundle_path = artifacts_dir / "xgb_refusal_bundle.pkl"

model_bundle = {
    "saved_at_utc": datetime.utcnow().isoformat() + "Z",
    "model_type": "xgboost.XGBClassifier",
    "xgb_model_json_path": str(xgb_json_path),
    "decision_threshold": float(best_t) if "best_t" in globals() else 0.5,
    "metrics": {
        "test_accuracy": float(test_accuracy) if "test_accuracy" in globals() else None,
        "test_precision": float(test_precision) if "test_precision" in globals() else None,
        "test_recall": float(test_recall) if "test_recall" in globals() else None,
        "test_f1": float(test_f1) if "test_f1" in globals() else None,
        "test_auc": float(test_auc) if "test_auc" in globals() else None,
    },
    # Needed to keep feature ordering consistent when preparing inference input
    "feature_columns": {
        "engineered": list(train_engineered_scaled_df.columns),
        "tfidf": list(train_tfidf_df.columns),
        "count": list(train_count_df.columns),
        "embeddings": list(train_embeddings_df.columns),
        "all": list(train_engineered_scaled_df.columns)
               + list(train_tfidf_df.columns)
               + list(train_count_df.columns)
               + list(train_embeddings_df.columns),
    },
    # Save preprocessor state that exists in this notebook
    "engineered_scaler": scaler_engineered,
}

with open(bundle_path, "wb") as f:
    pickle.dump(model_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved artifacts:")
print(f"- XGBoost JSON model: {xgb_json_path}")
print(f"- Python bundle (pickle): {bundle_path}")
print("\nNote: TF-IDF and Count vectorizer objects are not included because this notebook currently stores only their transformed DataFrames.")
print("For full raw-text inference portability, save fitted vectorizers too (or update pipeline.py to return them).")

Saved artifacts:
- XGBoost JSON model: artifacts\xgb_refusal_model.json
- Python bundle (pickle): artifacts\xgb_refusal_bundle.pkl

Note: TF-IDF and Count vectorizer objects are not included because this notebook currently stores only their transformed DataFrames.
For full raw-text inference portability, save fitted vectorizers too (or update pipeline.py to return them).
